<a href="https://colab.research.google.com/github/lucaslezcano03/Proyecto-IA/blob/main/dataset_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Sección 1: Instalación e Importación**
Configuración del entorno y librerías necesarias.

In [ ]:
# Instalación silenciosa de dependencias
!pip install langchain langchain-community --quiet

import pandas as pd
import langchain

print("✓ Librerías importadas correctamente.")

✓ Librerías importadas correctamente.


### **Sección 2: Carga del Dataset**
Carga inicial del archivo CSV con codificación compatible.

In [ ]:
archivo_original = 'sales_data_sample.csv'

try:
    df = pd.read_csv(archivo_original, encoding='unicode_escape')
    print(f"✓ Dataset original '{archivo_original}' cargado con éxito.")
    print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print("❌ Error: Sube 'sales_data_sample.csv' a la barra lateral de Colab.")

✓ Dataset original 'sales_data_sample.csv' cargado con éxito.
Dimensiones: 2823 filas y 25 columnas.


### **Sección 3: Normalización y Limpieza**
Tratamiento de valores nulos y conversión de tipos de datos.

In [ ]:
if 'df' in locals():
    # 1. Normalizar columnas de tipo Objeto (Texto)
    cols_object = df.select_dtypes(include=['object']).columns
    df[cols_object] = df[cols_object].fillna('N/A')

    # 2. Normalizar columnas numéricas (int/float)
    cols_numeric = df.select_dtypes(include=['number']).columns
    df[cols_numeric] = df[cols_numeric].fillna(0)

    # 3. Estandarización de fechas
    if 'ORDERDATE' in df.columns:
        df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])

    # VALIDACIÓN: Verificar si quedan nulos
    nulos_finales = df.isnull().sum().sum()
    print(f"✓ Proceso de normalización completado.")
    print(f"✓ Total de valores nulos en el dataset: {nulos_finales}")
    if nulos_finales == 0:
        print("✅ Validación exitosa: El dataset está 100% limpio.")

✓ Proceso de normalización completado.
✓ Total de valores nulos en el dataset: 0
✅ Validación exitosa: El dataset está 100% limpio.


### **Sección 4: Exportación y Verificación**
Guardado del archivo limpio y visualización de resultados.

In [ ]:
if 'df' in locals():
    # Nuevo nombre solicitado
    nombre_nuevo_archivo = 'ventas.csv'

    # Exportación
    df.to_csv(nombre_nuevo_archivo, index=False)
    print(f"✓ Archivo creado exitosamente como: {nombre_nuevo_archivo}")

    # Verificación de las primeras filas
    display(df.head(3))

✓ Archivo creado exitosamente como: ventas.csv


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium


# Sección 5: Inicialización del Agente Inteligente
# Configuración del modelo Mistral Small con capacidad de ejecución de código.

In [14]:
### **Sección 5: Inicialización del Agente Inteligente**
# Configuración del modelo Mistral Small con capacidad de ejecución de código.

!pip install -q langchain langchain-mistralai pandas langchain_experimental

from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import pandas as pd

# 1. Configuración de API y Modelo
api_key = userdata.get('MISTRAL_API_KEY')
llm = ChatMistralAI(api_key=api_key, model="mistral-small-latest")

# 2. Cargar el corpus normalizado
df = pd.read_csv('ventas.csv')

# 3. Prompt de precisión con contexto detallado del dataset
prefix_prompt = """
Eres un analista de datos experto en ventas. Tienes acceso a un dataframe de Pandas llamado 'df'.

Contexto de las columnas principales:
- ORDERNUMBER: Número único de identificación de la orden.
- QUANTITYORDERED: Cantidad de productos pedidos en esa línea.
- PRICEEACH: Precio de cada unidad del producto.
- SALES: Monto total de la venta (generalmente QUANTITYORDERED * PRICEEACH).
- ORDERDATE: Fecha en la que se realizó el pedido.
- STATUS: Estado actual (ej. Shipped, Cancelled, Resolved, On Hold).
- PRODUCTLINE: Categoría o línea del producto (ej. Motorcycles, Classic Cars).
- CUSTOMERNAME: Nombre del cliente o empresa.
- CITY y COUNTRY: Ubicación geográfica del cliente.

Instrucciones críticas:
1. Para cualquier pregunta numérica o estadística, DEBES generar y ejecutar código Python.
2. Si te preguntan por ventas totales, usa la columna 'SALES'.
3. Si hay valores 'N/A', ignóralos en cálculos matemáticos para no sesgar los resultados.
4. Responde siempre en español de forma clara y concisa.
"""

# 4. Creación del agente
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix_prompt,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling"
)

print("✓ Agente configurado con contexto de negocio.")

✓ Agente configurado con contexto de negocio.


In [15]:
# Prueba de funcionamiento
pregunta = "¿cuantas filas tiene el dataset?"
respuesta = agent.invoke(pregunta)
print(f"Respuesta del Agente: {respuesta['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


2823El dataset tiene **2,823 filas**.

> Finished chain.
Respuesta del Agente: El dataset tiene **2,823 filas**.


In [16]:
preguntas_prueba = [
    "¿Cuál fue el total de ventas (SALES) por cada año disponible?",
    "¿Cuáles son los 3 países que generaron más ingresos totales?",
    "¿Cuántos pedidos tienen un STATUS de 'Cancelled'?",
    "¿Cuál es el promedio de PRICEEACH para la línea 'Classic Cars'?"
]

for i, pregunta in enumerate(preguntas_prueba, 1):
    print(f"\n--- PRUEBA {i}: {pregunta} ---")
    try:
        resultado = agent.invoke(pregunta)
        print(f"\nRESPUESTA: {resultado['output']}")
    except Exception as e:
        print(f"Error en la prueba {i}: {str(e)}")


--- PRUEBA 1: ¿Cuál fue el total de ventas (SALES) por cada año disponible? ---


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Calcular el total de ventas (SALES) por cada año disponible en el dataframe\ntotal_ventas_por_año = df.groupby('YEAR_ID')['SALES'].sum().reset_index()\ntotal_ventas_por_año"}`


   YEAR_ID       SALES
0     2003  3516979.54
1     2004  4724162.60
2     2005  1791486.71El total de ventas (**SALES**) por cada año disponible es el siguiente:

| **Año** | **Total de Ventas** |
|---------|---------------------|
| 2003    | 3,516,979.54        |
| 2004    | 4,724,162.60        |
| 2005    | 1,791,486.71        |

> Finished chain.

RESPUESTA: El total de ventas (**SALES**) por cada año disponible es el siguiente:

| **Año** | **Total de Ventas** |
|---------|---------------------|
| 2003    | 3,516,979.54        |
| 2004    | 4,724,162.60        |
| 2005    | 1,791,486.71        |

--- PRUEBA 2: ¿Cuáles son los 3 países que g